In [2]:
import pandas as pd
import sqlite3
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

conn = sqlite3.connect("../data/nfl.db")
df_full = pd.read_sql_query("SELECT * FROM games_with_features", conn)
conn.close()
df_full['home_win'] = (df_full['home_score'] > df_full['away_score']).astype(int)

FEATURE_COLS = [
    'home_recent_form', 'away_recent_form',
    'home_recent_point_diff', 'away_recent_point_diff',
    'home_qb_recent_yards', 'away_qb_recent_yards',
    'home_qb_recent_tds', 'away_qb_recent_tds',
    'home_qb_recent_ints', 'away_qb_recent_ints',
    'home_qb_recent_epa', 'away_qb_recent_epa',
    'home_rb_recent_rush_yards', 'away_rb_recent_rush_yards',
    'home_rb_recent_rush_epa', 'away_rb_recent_rush_epa',
    'home_rb_recent_rec_yards', 'away_rb_recent_rec_yards',
    'home_wrte_recent_rec_yards', 'away_wrte_recent_rec_yards',
    'home_wrte_recent_rec_epa', 'away_wrte_recent_rec_epa',
    'home_wrte_recent_targets', 'away_wrte_recent_targets',
    'home_qb_injury_flag', 'away_qb_injury_flag',
    'home_rb_injury_flag', 'away_rb_injury_flag',
    'home_wrte_injury_flag', 'away_wrte_injury_flag',
    'home_epa_allowed_recent', 'away_epa_allowed_recent',
    'home_yards_allowed_recent', 'away_yards_allowed_recent',
    'home_takeaways_recent', 'away_takeaways_recent',
    'home_coach_h2h_wins', 'h2h_games_played',
    'home_elo_pre', 'away_elo_pre',
    'rest_advantage',
    'home_sack_rate_recent', 'away_sack_rate_recent',
    'home_pressure_pct_recent', 'away_pressure_pct_recent',
    'home_wr_height_advantage', 'away_wr_height_advantage',
    'home_wr_weight_advantage', 'away_wr_weight_advantage',
    'home_opp_cb_completion_allowed', 'away_opp_cb_completion_allowed',
    'home_opp_cb_rating_allowed', 'away_opp_cb_rating_allowed',
    'home_star_rb_injured', 'away_star_rb_injured',
    'home_star_wr_injured', 'away_star_wr_injured',
    'div_game'
]

y = df_full['home_win']
imputer = SimpleImputer(strategy='mean')
X = df_full[FEATURE_COLS].copy()
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=FEATURE_COLS, index=X.index)

train_mask = df_full['season'] <= 2023
test_mask = df_full['season'] >= 2024
X_train, X_test = X_imputed[train_mask], X_imputed[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

# L1-penalized logistic regression, auto-selects regularization strength via cross-validation
l1_model = make_pipeline(
    StandardScaler(),
    LogisticRegressionCV(cv=5, penalty='l1', solver='liblinear', Cs=10, max_iter=2000)
)
l1_model.fit(X_train, y_train)
acc = l1_model.score(X_test, y_test)

print(f"L1-regularized logistic regression test accuracy: {acc:.3f}")

coefs = l1_model.named_steps['logisticregressioncv'].coef_[0]
zeroed_out = [f for f, c in zip(FEATURE_COLS, coefs) if abs(c) < 0.001]
print(f"\nFeatures shrunk to ~zero ({len(zeroed_out)} of {len(FEATURE_COLS)}):")
print(zeroed_out)

c:\Users\drdre\Projects\sports_betting\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\drdre\Projects\sports_betting\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2123: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
c:\Users\drdre\Projects\sports_betting\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2137: Futur

L1-regularized logistic regression test accuracy: 0.679

Features shrunk to ~zero (0 of 58):
[]


In [3]:
best_C = l1_model.named_steps['logisticregressioncv'].C_
print(f"Best regularization strength (C) chosen via CV: {best_C}")

coef_df = pd.DataFrame({'feature': FEATURE_COLS, 'coefficient': coefs})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df = coef_df.sort_values('abs_coef')

print("\nSmallest 10 coefficients (weakest contributors):")
print(coef_df.head(10))

Best regularization strength (C) chosen via CV: [166.81005372]

Smallest 10 coefficients (weakest contributors):
                       feature  coefficient  abs_coef
10          home_qb_recent_epa    -0.001145  0.001145
27         away_rb_injury_flag    -0.001727  0.001727
18  home_wrte_recent_rec_yards     0.004389  0.004389
1             away_recent_form     0.009420  0.009420
22    home_wrte_recent_targets    -0.012039  0.012039
36         home_coach_h2h_wins     0.012216  0.012216
15     away_rb_recent_rush_epa    -0.012892  0.012892
19  away_wrte_recent_rec_yards     0.014662  0.014662
53        home_star_rb_injured    -0.016526  0.016526
11          away_qb_recent_epa    -0.019899  0.019899


In [9]:
import numpy as np

elo_games_mov = df_full[['game_id', 'season', 'gameday', 'home_team_std', 'away_team_std', 'home_win', 'home_score', 'away_score']].copy()
elo_games_mov = elo_games_mov.sort_values(['season', 'gameday']).reset_index(drop=True)

all_teams = set(elo_games_mov['home_team_std']).union(set(elo_games_mov['away_team_std']))
elo_mov = {team: 1500 for team in all_teams}
k_factor, home_advantage, revert_fraction = 20, 65, 1/3
current_season = None

home_elo_pre_mov = []
away_elo_pre_mov = []

for _, row in elo_games_mov.iterrows():
    if current_season is not None and row['season'] != current_season:
        for team in elo_mov:
            elo_mov[team] = elo_mov[team] * (1 - revert_fraction) + 1500 * revert_fraction
    current_season = row['season']

    h, a = row['home_team_std'], row['away_team_std']
    home_elo_pre_mov.append(elo_mov[h])
    away_elo_pre_mov.append(elo_mov[a])

    elo_diff_pre = (elo_mov[h] + home_advantage) - elo_mov[a]
    expected_home = 1 / (1 + 10 ** (-elo_diff_pre / 400))
    actual_home = row['home_win']

    margin = abs(row['home_score'] - row['away_score'])
    winner_elo_diff = elo_diff_pre if actual_home == 1 else -elo_diff_pre
    mov_multiplier = np.log(margin + 1) * (2.2 / (winner_elo_diff * 0.001 + 2.2))

    elo_mov[h] += k_factor * mov_multiplier * (actual_home - expected_home)
    elo_mov[a] += k_factor * mov_multiplier * ((1 - actual_home) - (1 - expected_home))

elo_games_mov['home_elo_mov_pre'] = home_elo_pre_mov
elo_games_mov['away_elo_mov_pre'] = away_elo_pre_mov

print(elo_games_mov[['game_id', 'home_team_std', 'away_team_std', 'home_elo_mov_pre', 'away_elo_mov_pre']].tail(10))

              game_id home_team_std away_team_std  home_elo_mov_pre  \
3018   2025_19_SF_PHI           PHI            SF       1601.142266   
3019   2025_19_LAC_NE            NE           LAC       1601.265178   
3020  2025_19_HOU_PIT           PIT           HOU       1535.832599   
3021  2025_20_BUF_DEN           DEN           BUF       1661.989688   
3022   2025_20_SF_SEA           SEA            SF       1706.903989   
3023   2025_20_LA_CHI           CHI            LA       1551.383956   
3024   2025_20_HOU_NE            NE           HOU       1618.548104   
3025   2025_21_NE_DEN           DEN            NE       1673.202571   
3026   2025_21_LA_SEA           SEA            LA       1725.380582   
3027   2025_22_SEA_NE            NE           SEA       1662.476934   

      away_elo_mov_pre  
3018       1584.953337  
3019       1554.220641  
3020       1656.832925  
3021       1667.588939  
3022       1605.499917  
3023       1661.906145  
3024       1683.206572  
3025       1644.16

In [12]:
elo_split_games = df_full[['game_id', 'season', 'gameday', 'home_team_std', 'away_team_std',
                             'home_score', 'away_score']].copy()
elo_split_games = elo_split_games.sort_values(['season', 'gameday']).reset_index(drop=True)

all_teams = set(elo_split_games['home_team_std']).union(set(elo_split_games['away_team_std']))
league_avg_points = elo_split_games[['home_score', 'away_score']].values.mean()  # baseline expected points

off_rating = {team: 0.0 for team in all_teams}  # 0 = league average offense
def_rating = {team: 0.0 for team in all_teams}  # 0 = league average defense (positive = good defense, allows fewer)

k_off_def = 0.05  # learning rate for points-based updates, tuned separately from win/loss Elo's k=20
revert_fraction = 1/3
current_season = None

home_off_pre, home_def_pre, away_off_pre, away_def_pre = [], [], [], []

for _, row in elo_split_games.iterrows():
    if current_season is not None and row['season'] != current_season:
        for team in off_rating:
            off_rating[team] *= (1 - revert_fraction)
            def_rating[team] *= (1 - revert_fraction)
    current_season = row['season']

    h, a = row['home_team_std'], row['away_team_std']
    home_off_pre.append(off_rating[h])
    home_def_pre.append(def_rating[h])
    away_off_pre.append(off_rating[a])
    away_def_pre.append(def_rating[a])

    # Expected points = league average, adjusted by offense's rating and opponent defense's rating
    expected_home_score = league_avg_points + off_rating[h] - def_rating[a]
    expected_away_score = league_avg_points + off_rating[a] - def_rating[h]

    # Update: actual vs expected, in both directions (offense AND the opponent's defense)
    off_rating[h] += k_off_def * (row['home_score'] - expected_home_score)
    def_rating[a] -= k_off_def * (row['home_score'] - expected_home_score)  # opponent D gets blamed/credited too

    off_rating[a] += k_off_def * (row['away_score'] - expected_away_score)
    def_rating[h] -= k_off_def * (row['away_score'] - expected_away_score)

elo_split_games['home_off_rating_pre'] = home_off_pre
elo_split_games['home_def_rating_pre'] = home_def_pre
elo_split_games['away_off_rating_pre'] = away_off_pre
elo_split_games['away_def_rating_pre'] = away_def_pre

print(f"League average points per team per game: {league_avg_points:.2f}")
elo_split_games[['game_id', 'home_team_std', 'away_team_std', 'home_off_rating_pre', 'home_def_rating_pre']].tail(10)

League average points per team per game: 22.83


,game_id,home_team_std,away_team_std,home_off_rating_pre,home_def_rating_pre
3018,2025_19_SF_PHI,PHI,SF,0.556482,3.638103
3019,2025_19_LAC_NE,NE,LAC,2.243991,1.952572
3020,2025_19_HOU_PIT,PIT,HOU,-0.434345,1.241064
3021,2025_20_BUF_DEN,DEN,BUF,0.728187,3.136743
3022,2025_20_SF_SEA,SEA,SF,2.908015,3.948497
3023,2025_20_LA_CHI,CHI,LA,1.274373,0.059561
3024,2025_20_HOU_NE,NE,HOU,1.896208,2.803345
3025,2025_21_NE_DEN,DEN,NE,1.252833,2.857970
3026,2025_21_LA_SEA,SEA,LA,3.677495,4.725146
3027,2025_22_SEA_NE,NE,SEA,1.643936,3.758737


In [13]:
print("Offense rating range:", min(off_rating.values()), "to", max(off_rating.values()))
print("Defense rating range:", min(def_rating.values()), "to", max(def_rating.values()))
print()

cle_def_check = pd.concat([
    elo_split_games[elo_split_games['home_team_std']=='CLE'][['season','gameday','home_def_rating_pre']].rename(columns={'home_def_rating_pre':'def_rating'}),
    elo_split_games[elo_split_games['away_team_std']=='CLE'][['season','gameday','away_def_rating_pre']].rename(columns={'away_def_rating_pre':'def_rating'})
]).sort_values('gameday')

print(cle_def_check[cle_def_check['season']==2016].tail(3))

Offense rating range: -5.73224158096026 to 5.011868743142617
Defense rating range: -4.794830127109257 to 4.86352353958439

     season     gameday  def_rating
478    2016  2016-12-18   -3.697171
495    2016  2016-12-24   -3.946948
513    2016  2017-01-01   -3.430795


In [14]:
split_elo_merge = elo_split_games[['game_id', 'home_off_rating_pre', 'home_def_rating_pre',
                                     'away_off_rating_pre', 'away_def_rating_pre']]
df_full = df_full.drop(columns=['home_off_rating_pre', 'home_def_rating_pre',
                                  'away_off_rating_pre', 'away_def_rating_pre'], errors='ignore')
df_full = df_full.merge(split_elo_merge, on='game_id', how='left')

# Test ADDING split O/D ratings alongside your existing combined MOV Elo
split_elo_features = FEATURE_COLS + ['home_off_rating_pre', 'home_def_rating_pre',
                                       'away_off_rating_pre', 'away_def_rating_pre']

X_split = df_full[split_elo_features].copy()
X_split_imputed = pd.DataFrame(imputer.fit_transform(X_split), columns=split_elo_features, index=X_split.index)
X_split_train, X_split_test = X_split_imputed[train_mask], X_split_imputed[test_mask]

model_split = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_split.fit(X_split_train, y_train)
acc_split = model_split.score(X_split_test, y_test)

print(f"Baseline (MOV Elo only): 0.679")
print(f"+ Split offense/defense Elo (alongside): {acc_split:.3f}")

Baseline (MOV Elo only): 0.679
+ Split offense/defense Elo (alongside): 0.670


In [15]:
split_replace_features = [f for f in FEATURE_COLS if f not in ['home_elo_pre', 'away_elo_pre']] + [
    'home_off_rating_pre', 'home_def_rating_pre', 'away_off_rating_pre', 'away_def_rating_pre']

X_replace = df_full[split_replace_features].copy()
X_replace_imputed = pd.DataFrame(imputer.fit_transform(X_replace), columns=split_replace_features, index=X_replace.index)
X_replace_train, X_replace_test = X_replace_imputed[train_mask], X_replace_imputed[test_mask]

model_replace = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_replace.fit(X_replace_train, y_train)
acc_replace = model_replace.score(X_replace_test, y_test)

print(f"Baseline (MOV Elo only): 0.679")
print(f"Split O/D Elo REPLACING combined Elo: {acc_replace:.3f}")

Baseline (MOV Elo only): 0.679
Split O/D Elo REPLACING combined Elo: 0.682


In [16]:
df_full['home_off_vs_away_def'] = df_full['home_off_rating_pre'] - df_full['away_def_rating_pre']
df_full['away_off_vs_home_def'] = df_full['away_off_rating_pre'] - df_full['home_def_rating_pre']

interaction_features = FEATURE_COLS + ['home_off_vs_away_def', 'away_off_vs_home_def']

X_int = df_full[interaction_features].copy()
X_int_imputed = pd.DataFrame(imputer.fit_transform(X_int), columns=interaction_features, index=X_int.index)
X_int_train, X_int_test = X_int_imputed[train_mask], X_int_imputed[test_mask]

model_int = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_int.fit(X_int_train, y_train)
acc_int = model_int.score(X_int_test, y_test)

print(f"Baseline (split O/D Elo): 0.682")
print(f"+ Matchup interaction terms: {acc_int:.3f}")

Baseline (split O/D Elo): 0.682
+ Matchup interaction terms: 0.668


In [17]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)

X_base = df_full[FEATURE_COLS].copy()
X_base_imputed = pd.DataFrame(imputer.fit_transform(X_base), columns=FEATURE_COLS, index=X_base.index)
X_base_train, X_base_test = X_base_imputed[train_mask], X_base_imputed[test_mask]

logreg_final = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg_final.fit(X_base_train, y_train)
xgb_model.fit(X_base_train, y_train)

logreg_probs = logreg_final.predict_proba(X_base_test)[:, 1]
xgb_probs = xgb_model.predict_proba(X_base_test)[:, 1]

ensemble_probs = (logreg_probs + xgb_probs) / 2
ensemble_preds = (ensemble_probs > 0.5).astype(int)
ensemble_acc = (ensemble_preds == y_test.values).mean()

print(f"Logistic regression alone: {logreg_final.score(X_base_test, y_test):.3f}")
print(f"XGBoost alone: {xgb_model.score(X_base_test, y_test):.3f}")
print(f"Simple average ensemble: {ensemble_acc:.3f}")

Logistic regression alone: 0.677
XGBoost alone: 0.646
Simple average ensemble: 0.675


In [18]:
FEATURE_COLS = [f for f in FEATURE_COLS if f not in ['home_elo_pre', 'away_elo_pre']] + [
    'home_off_rating_pre', 'home_def_rating_pre', 'away_off_rating_pre', 'away_def_rating_pre'
]

print(len(FEATURE_COLS))

60


In [19]:
X_base = df_full[FEATURE_COLS].copy()
X_base_imputed = pd.DataFrame(imputer.fit_transform(X_base), columns=FEATURE_COLS, index=X_base.index)
X_base_train, X_base_test = X_base_imputed[train_mask], X_base_imputed[test_mask]

logreg_final = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg_final.fit(X_base_train, y_train)
xgb_model.fit(X_base_train, y_train)

logreg_probs = logreg_final.predict_proba(X_base_test)[:, 1]
xgb_probs = xgb_model.predict_proba(X_base_test)[:, 1]

ensemble_probs = (logreg_probs + xgb_probs) / 2
ensemble_preds = (ensemble_probs > 0.5).astype(int)
ensemble_acc = (ensemble_preds == y_test.values).mean()

print(f"Logistic regression alone: {logreg_final.score(X_base_test, y_test):.3f}")
print(f"XGBoost alone: {xgb_model.score(X_base_test, y_test):.3f}")
print(f"Simple average ensemble: {ensemble_acc:.3f}")

Logistic regression alone: 0.682
XGBoost alone: 0.618
Simple average ensemble: 0.644


In [25]:
import pandas as pd
import sqlite3
import nflreadpy as nfl

conn = sqlite3.connect("../data/nfl.db")
df_full = pd.read_sql_query("SELECT * FROM games_with_features", conn)
conn.close()
df_full['home_win'] = (df_full['home_score'] > df_full['away_score']).astype(int)

game_dates = df_full[['game_id', 'gameday']].drop_duplicates()

print(f"Loaded {len(df_full)} games")

Loaded 3028 games


In [27]:
ngs_full = nfl.load_nextgen_stats(seasons=list(range(2016, 2026)), stat_type='passing').to_pandas()
ngs_full = ngs_full[ngs_full['week'] > 0]
ngs_full = ngs_full[['player_gsis_id', 'team_abbr', 'season', 'week',
                      'completion_percentage_above_expectation']].rename(
    columns={'player_gsis_id': 'player_id', 'team_abbr': 'team', 'completion_percentage_above_expectation': 'cpoe'})

# Build a team-level game lookup: each team's game_id for every season/week they played
home_games = df_full[['season', 'week', 'game_id', 'home_team_std']].rename(columns={'home_team_std': 'team'})
away_games = df_full[['season', 'week', 'game_id', 'away_team_std']].rename(columns={'away_team_std': 'team'})
team_game_lookup = pd.concat([home_games, away_games], ignore_index=True)

ngs_full = ngs_full.merge(team_game_lookup, on=['team', 'season', 'week'], how='left')
ngs_full = ngs_full.merge(game_dates, on='game_id', how='left')
ngs_full = ngs_full.dropna(subset=['game_id']).sort_values(['player_id', 'gameday']).reset_index(drop=True)

ngs_full['recent_cpoe'] = (
    ngs_full.groupby('player_id')['cpoe']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

print(len(ngs_full))
ngs_full[['player_id', 'team', 'season', 'week', 'cpoe', 'recent_cpoe']].head(10)

5327


,player_id,team,season,week,cpoe,recent_cpoe
0,00-0019596,NE,2016,5,2.547140,NaN
1,00-0019596,NE,2016,6,8.379676,2.547140
2,00-0019596,NE,2016,7,2.616260,5.463408
3,00-0019596,NE,2016,8,2.928208,4.514359
4,00-0019596,NE,2016,10,10.537489,4.117821
5,00-0019596,NE,2016,11,-5.112689,5.401755
6,00-0019596,NE,2016,12,-5.515540,3.869789
7,00-0019596,NE,2016,13,8.351488,1.090746
8,00-0019596,NE,2016,14,4.859212,2.237791
9,00-0019596,NE,2016,15,-6.506051,2.623992


In [29]:
def build_team_rolling(df, value_col, position_filter=None):
    d = df[['team_abbr', 'season', 'week', value_col]].copy()
    d = d[d['week'] > 0].rename(columns={'team_abbr': 'team'})
    d = d.groupby(['team', 'season', 'week'], as_index=False)[value_col].mean()
    d = d.merge(team_game_lookup, on=['team', 'season', 'week'], how='left')
    d = d.merge(game_dates, on='game_id', how='left').dropna(subset=['game_id'])
    d = d.sort_values(['team', 'gameday']).reset_index(drop=True)
    d[f'recent_{value_col}'] = d.groupby('team')[value_col].transform(
        lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    return d[['team', 'game_id', f'recent_{value_col}']]

ngs_pass = nfl.load_nextgen_stats(seasons=list(range(2016, 2026)), stat_type='passing').to_pandas()
ngs_rec = nfl.load_nextgen_stats(seasons=list(range(2016, 2026)), stat_type='receiving').to_pandas()
ngs_rush = nfl.load_nextgen_stats(seasons=list(range(2016, 2026)), stat_type='rushing').to_pandas()

candidates = {
    'time_to_throw': build_team_rolling(ngs_pass, 'avg_time_to_throw'),
    'aggressiveness': build_team_rolling(ngs_pass, 'aggressiveness'),
    'separation': build_team_rolling(ngs_rec, 'avg_separation'),
    'yac_above_exp': build_team_rolling(ngs_rec, 'avg_yac_above_expectation'),
    'ryoe_per_att': build_team_rolling(ngs_rush, 'rush_yards_over_expected_per_att'),
}

df_check = df_full.copy()
for name, tbl in candidates.items():
    col = tbl.columns[-1]
    home_tbl = tbl.rename(columns={'team': 'home_team_std', col: f'home_{name}'})
    away_tbl = tbl.rename(columns={'team': 'away_team_std', col: f'away_{name}'})
    df_check = df_check.merge(home_tbl, on=['home_team_std', 'game_id'], how='left')
    df_check = df_check.merge(away_tbl, on=['away_team_std', 'game_id'], how='left')

corr_cols = [f'home_{n}' for n in candidates] + [f'away_{n}' for n in candidates]
print(df_check[corr_cols + ['home_win']].corr()['home_win'].drop('home_win').sort_values(key=abs, ascending=False))

away_aggressiveness    0.084781
away_yac_above_exp    -0.069868
away_separation       -0.056748
home_aggressiveness   -0.052969
home_yac_above_exp     0.051413
home_ryoe_per_att      0.040212
home_time_to_throw     0.026333
away_time_to_throw    -0.024363
away_ryoe_per_att      0.013876
home_separation        0.012180
Name: home_win, dtype: float64


In [30]:
agg_tbl = candidates['aggressiveness']
home_agg = agg_tbl.rename(columns={'team': 'home_team_std', 'recent_aggressiveness': 'home_aggressiveness'})
away_agg = agg_tbl.rename(columns={'team': 'away_team_std', 'recent_aggressiveness': 'away_aggressiveness'})

df_full = df_full.drop(columns=['home_aggressiveness', 'away_aggressiveness'], errors='ignore')
df_full = df_full.merge(home_agg, on=['home_team_std', 'game_id'], how='left')
df_full = df_full.merge(away_agg, on=['away_team_std', 'game_id'], how='left')

agg_features = FEATURE_COLS + ['home_aggressiveness', 'away_aggressiveness']
X_agg = df_full[agg_features].copy()
X_agg_imputed = pd.DataFrame(imputer.fit_transform(X_agg), columns=agg_features, index=X_agg.index)
X_agg_train, X_agg_test = X_agg_imputed[train_mask], X_agg_imputed[test_mask]

model_agg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_agg.fit(X_agg_train, y_train)
acc_agg = model_agg.score(X_agg_test, y_test)

print(f"Baseline: 0.682")
print(f"+ QB aggressiveness: {acc_agg:.3f}")

Baseline: 0.682
+ QB aggressiveness: 0.681
